# 08 - Buffered Callaway-Sant'Anna ATT

This notebook estimates the main direct effect using a library-compatible buffered donor pool. It uses `diff-diff` as the main implementation and compares the no-covariate specification with `csdid`.

In [ ]:
from pathlib import Path
import json
import sys
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'notebooks').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from utils.estimation_workflow_utils import append_tag_to_filename, build_stable_buffered_did_panel, hac_for_group_time_effects

DATA_DIR = PROJECT_ROOT / 'data'
INTERMEDIATE_DIR = DATA_DIR / 'intermediate'
SPATIAL_DIR = INTERMEDIATE_DIR / 'spatial_structure'
OUTPUT_DIR = PROJECT_ROOT / 'outputs'
TABLE_DIR = OUTPUT_DIR / 'tables'
FIG_DIR = OUTPUT_DIR / 'figures'
for path in [INTERMEDIATE_DIR, TABLE_DIR, FIG_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print('Project root:', PROJECT_ROOT)


## User configuration

In [ ]:
RUN_TAG = '1km'
START_YEAR = 2001
END_YEAR = None
TREATMENT_KEY = 'protected_area'
CELL_COL = 'cell_id'
YEAR_COL = 'year'
OUTCOME_COL = 'loss_m2'
MAIN_BUFFER_KM = 25
BUFFER_RADII_KM = [10, 25, 50]

PANEL_PATH = INTERMEDIATE_DIR / f'panel_treatment_{RUN_TAG}.parquet'
if not PANEL_PATH.exists():
    PANEL_PATH = INTERMEDIATE_DIR / 'panel_treatment.parquet'
EXPOSURE_PATH = INTERMEDIATE_DIR / append_tag_to_filename('06_nearest_treated_exposure.parquet', RUN_TAG)
CENTROID_PATH = SPATIAL_DIR / append_tag_to_filename('03_grid_centroids.parquet', RUN_TAG)
if not CENTROID_PATH.exists():
    CENTROID_PATH = SPATIAL_DIR / '03_grid_centroids.parquet'
USE_CORE_COVARIATES = False
USE_ROBUST_COVARIATES = False
CORE_COVARIATE_PATH = INTERMEDIATE_DIR / append_tag_to_filename('panel_core_covariates.parquet', RUN_TAG)
ROBUST_COVARIATE_PATH = INTERMEDIATE_DIR / append_tag_to_filename('panel_robustness_covariates.parquet', RUN_TAG)

ESTIMATION_SAMPLE_NAME = '08_buffered_estimation_panel.parquet'
DIFFDIFF_RESULTS_NAME = '08_diffdiff_cs_results.csv'
CSDID_RESULTS_NAME = '08_csdid_comparison_results.csv'
COMPARISON_NAME = '08_diffdiff_csdid_comparison.csv'
SPATIAL_HAC_NAME = '08_spatial_hac_attgt.csv'
EVENT_FIG_NAME = '08_diffdiff_event_study.png'

# Optional exploratory DDD. This is not staggered; it collapses the design to a binary post period.
RUN_TRIPLE_DIFFERENCE = False
DDD_POST_YEAR = 2017
DDD_GROUP_COL = 'ever_protected_area'
DDD_PARTITION_COL = 'ever_land_title_any'
DDD_RESULTS_NAME = '08_exploratory_triple_difference.csv'

HAC_CUTOFF_KM = 25
HAC_KERNEL = 'bartlett'
HAC_MAX_PAIRS = 50_000_000


## Load and build buffered estimation sample

The library-compatible sample keeps treated cohorts with pre-treatment observations and uses never-treated controls that remain outside the chosen buffer throughout the analysis window.

In [ ]:
import pyarrow.parquet as pq

first_treat_col = 'first_treat_year'
panel_schema_cols = pq.ParquetFile(PANEL_PATH).schema_arrow.names
panel_cols = [
    CELL_COL, YEAR_COL, OUTCOME_COL, first_treat_col,
    'treated_before_panel_start', 'never_treated', 'ever_treated',
    DDD_GROUP_COL, DDD_PARTITION_COL,
]
panel_cols = [c for c in panel_cols if c in panel_schema_cols]
panel = pd.read_parquet(PANEL_PATH, columns=panel_cols)
panel[CELL_COL] = panel[CELL_COL].astype('string')
panel[YEAR_COL] = pd.to_numeric(panel[YEAR_COL], errors='coerce').astype(int)
exposure = pd.read_parquet(EXPOSURE_PATH)
exposure[CELL_COL] = exposure[CELL_COL].astype('string')
exposure[YEAR_COL] = pd.to_numeric(exposure[YEAR_COL], errors='coerce').astype(int)
centroids = pd.read_parquet(CENTROID_PATH)
centroids[CELL_COL] = centroids[CELL_COL].astype('string')

did_panel = build_stable_buffered_did_panel(
    panel,
    exposure,
    buffer_km=MAIN_BUFFER_KM,
    cell_col=CELL_COL,
    year_col=YEAR_COL,
    first_treat_col=first_treat_col,
    treated_before_panel_col='treated_before_panel_start',
    outcome_col=OUTCOME_COL,
    min_year=START_YEAR,
    max_year=END_YEAR,
)
sample_path = INTERMEDIATE_DIR / append_tag_to_filename(ESTIMATION_SAMPLE_NAME, RUN_TAG)
did_panel.to_parquet(sample_path, index=False)
print('Saved:', sample_path)
print('Rows:', f'{len(did_panel):,}')
print('Cells:', f'{did_panel[CELL_COL].nunique():,}')
print('Treated cohorts:', sorted([int(x) for x in did_panel['first_treat_for_did'].unique() if x > 0])[:10], '...')


## Attach covariate sets

In [ ]:
covariate_sets = {'none': []}
analysis_panel = did_panel.copy()

if USE_CORE_COVARIATES and CORE_COVARIATE_PATH.exists():
    core = pd.read_parquet(CORE_COVARIATE_PATH)
    core[CELL_COL] = core[CELL_COL].astype('string')
    core_covariates = [c for c in core.columns if c not in {CELL_COL, 'cell_lon', 'cell_lat', 'first_treat_year', 'ever_treated', 'never_treated'}]
    analysis_panel = analysis_panel.merge(core[[CELL_COL] + core_covariates], on=CELL_COL, how='left')
    covariate_sets['core'] = core_covariates

if USE_ROBUST_COVARIATES and ROBUST_COVARIATE_PATH.exists():
    robust = pd.read_parquet(ROBUST_COVARIATE_PATH)
    robust[CELL_COL] = robust[CELL_COL].astype('string')
    robust_covariates = [c for c in robust.columns if c not in {CELL_COL, 'cell_lon', 'cell_lat', 'first_treat_year', 'ever_treated', 'never_treated'}]
    analysis_panel = analysis_panel.merge(robust[[CELL_COL] + robust_covariates], on=CELL_COL, how='left')
    covariate_sets['robust'] = robust_covariates

print('Covariate paths:')
print(' - core:', CORE_COVARIATE_PATH, '| enabled:', USE_CORE_COVARIATES)
print(' - robust:', ROBUST_COVARIATE_PATH, '| enabled:', USE_ROBUST_COVARIATES)
print('Covariate sets:', {k: len(v) for k, v in covariate_sets.items()})


## Estimate with diff-diff

In [ ]:
try:
    from diff_diff import CallawaySantAnna, plot_event_study
except ImportError as exc:
    raise ImportError('Install estimator dependencies with `%pip install -r ../requirements.txt` before running notebook 08.') from exc

def extract_diffdiff_results(result, covariate_set):
    rows = []
    if hasattr(result, 'group_time_effects'):
        effects = result.group_time_effects
    elif hasattr(result, 'results_') and hasattr(result.results_, 'group_time_effects'):
        effects = result.results_.group_time_effects
    else:
        effects = []
    for eff in effects:
        rows.append({
            'covariate_set': covariate_set,
            'group': getattr(eff, 'group', np.nan),
            'time': getattr(eff, 'time', np.nan),
            'att': getattr(eff, 'att', getattr(eff, 'estimate', np.nan)),
            'se': getattr(eff, 'se', getattr(eff, 'std_error', np.nan)),
        })
    overall = getattr(result, 'overall_att', np.nan)
    overall_se = getattr(result, 'overall_se', np.nan)
    return pd.DataFrame(rows), {'covariate_set': covariate_set, 'overall_att': overall, 'overall_se': overall_se}

diffdiff_rows = []
overall_rows = []
diffdiff_results = {}
for cov_name, covs in covariate_sets.items():
    model_df = analysis_panel[[CELL_COL, YEAR_COL, OUTCOME_COL, 'first_treat_for_did'] + covs].dropna().copy()
    method = 'dr' if covs else 'reg'
    estimator = CallawaySantAnna(control_group='never_treated', estimation_method=method, n_bootstrap=0, seed=123)
    fit_kwargs = dict(data=model_df, outcome=OUTCOME_COL, unit=CELL_COL, time=YEAR_COL, first_treat='first_treat_for_did')
    if covs:
        fit_kwargs['covariates'] = covs
    result = estimator.fit(**fit_kwargs)
    diffdiff_results[cov_name] = result
    gt, overall = extract_diffdiff_results(result, cov_name)
    diffdiff_rows.append(gt)
    overall_rows.append(overall)

diffdiff_gt = pd.concat(diffdiff_rows, ignore_index=True) if diffdiff_rows else pd.DataFrame()
diffdiff_overall = pd.DataFrame(overall_rows)
diffdiff_path = TABLE_DIR / append_tag_to_filename(DIFFDIFF_RESULTS_NAME, RUN_TAG)
diffdiff_gt.to_csv(diffdiff_path, index=False)
print('Saved:', diffdiff_path)
print(diffdiff_overall.to_string(index=False))


## Compare no-covariate result with csdid

In [ ]:
try:
    from csdid.att_gt import ATTgt
    csdid_available = True
except ImportError:
    csdid_available = False
    warnings.warn('csdid is not installed; skipping cross-check.')

csdid_gt = pd.DataFrame()
if csdid_available:
    cs_df = analysis_panel[[CELL_COL, YEAR_COL, OUTCOME_COL, 'first_treat_for_did']].copy()
    cs_df['unit_id_num'] = pd.factorize(cs_df[CELL_COL])[0] + 1
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        att = ATTgt(
            yname=OUTCOME_COL,
            tname=YEAR_COL,
            idname='unit_id_num',
            gname='first_treat_for_did',
            xformla=f'{OUTCOME_COL} ~ 1',
            data=cs_df,
            bstrap=False,
        ).fit()
    csdid_gt = att.summ_attgt().summary2.copy()
    csdid_path = TABLE_DIR / append_tag_to_filename(CSDID_RESULTS_NAME, RUN_TAG)
    csdid_gt.to_csv(csdid_path, index=False)
    print('Saved:', csdid_path)
    print(csdid_gt.head().to_string(index=False))
else:
    print('csdid comparison skipped.')


## Spatial HAC standard errors

This block recomputes no-covariate 2x2 ATT(g,t) influence contributions for the same buffered sample and applies a Conley-style spatial HAC estimator. It is the HAC inference layer for the main no-covariate specification; covariate-adjusted HAC requires extracting or recreating covariate-adjusted influence functions.

In [ ]:
spatial_hac = pd.DataFrame()
if not diffdiff_gt.empty:
    effects_for_hac = diffdiff_gt.loc[diffdiff_gt['covariate_set'] == 'none', ['group', 'time']].dropna().drop_duplicates()
    effects_for_hac['group'] = pd.to_numeric(effects_for_hac['group'], errors='coerce')
    effects_for_hac['time'] = pd.to_numeric(effects_for_hac['time'], errors='coerce')
    effects_for_hac = effects_for_hac.dropna().astype({'group': int, 'time': int})
    spatial_hac = hac_for_group_time_effects(
        analysis_panel,
        centroids,
        cell_col=CELL_COL,
        year_col=YEAR_COL,
        outcome_col=OUTCOME_COL,
        group_col='first_treat_for_did',
        effects=effects_for_hac,
        cutoff_km=HAC_CUTOFF_KM,
        kernel=HAC_KERNEL,
        max_pairs=HAC_MAX_PAIRS,
    )
    spatial_hac_path = TABLE_DIR / append_tag_to_filename(SPATIAL_HAC_NAME, RUN_TAG)
    spatial_hac.to_csv(spatial_hac_path, index=False)
    print('Saved:', spatial_hac_path)
    print(spatial_hac.head(12).to_string(index=False))
    if spatial_hac['hac_truncated'].any():
        print('Warning: at least one HAC calculation hit HAC_MAX_PAIRS. Increase HAC_MAX_PAIRS or reduce HAC_CUTOFF_KM after checking runtime/memory.')
else:
    print('Spatial HAC skipped because diff-diff group-time results are empty.')


## Optional exploratory triple difference

`diff-diff`'s triple-difference estimator is a binary group x partition x post design. This block is off by default because it is not a staggered C&S replacement. Use it as a diagnostic after choosing a meaningful binary post period.

In [ ]:
ddd_results = pd.DataFrame()
if RUN_TRIPLE_DIFFERENCE:
    try:
        from diff_diff import TripleDifference
    except ImportError as exc:
        raise ImportError('Install estimator dependencies with `%pip install -r ../requirements.txt` before running the DDD block.') from exc

    required_ddd_cols = {DDD_GROUP_COL, DDD_PARTITION_COL}
    missing_ddd_cols = sorted(required_ddd_cols - set(panel.columns))
    if missing_ddd_cols:
        raise ValueError(f'Missing DDD columns in panel: {missing_ddd_cols}')
    ddd_df = panel[[CELL_COL, YEAR_COL, OUTCOME_COL, DDD_GROUP_COL, DDD_PARTITION_COL]].dropna().copy()
    ddd_df['post_period'] = (ddd_df[YEAR_COL] >= DDD_POST_YEAR).astype(int)
    ddd_df['ddd_group'] = ddd_df[DDD_GROUP_COL].astype(int)
    ddd_df['ddd_partition'] = ddd_df[DDD_PARTITION_COL].astype(int)

    ddd = TripleDifference()
    ddd_fit = ddd.fit(
        data=ddd_df,
        outcome=OUTCOME_COL,
        unit=CELL_COL,
        time=YEAR_COL,
        treatment='ddd_group',
        post='post_period',
        partition='ddd_partition',
    )
    ddd_results = pd.DataFrame([{
        'post_year': DDD_POST_YEAR,
        'group_col': DDD_GROUP_COL,
        'partition_col': DDD_PARTITION_COL,
        'ddd_estimate': getattr(ddd_fit, 'ddd_estimate', getattr(ddd_fit, 'effect', np.nan)),
        'standard_error': getattr(ddd_fit, 'standard_error', getattr(ddd_fit, 'se', np.nan)),
    }])
    ddd_path = TABLE_DIR / append_tag_to_filename(DDD_RESULTS_NAME, RUN_TAG)
    ddd_results.to_csv(ddd_path, index=False)
    print('Saved:', ddd_path)
    print(ddd_results.to_string(index=False))
else:
    print('Triple-difference block skipped. Set RUN_TRIPLE_DIFFERENCE = True after choosing DDD_POST_YEAR and the group/partition definitions.')


## Comparison table and event-study figure

In [ ]:
comparison = pd.DataFrame()
if not diffdiff_gt.empty and not csdid_gt.empty:
    dd = diffdiff_gt[diffdiff_gt['covariate_set'] == 'none'].copy()
    cs = csdid_gt.rename(columns={'Group': 'group', 'Time': 'time', 'ATT(g, t)': 'att_csdid', 'Std. Error': 'se_csdid'})
    for col in ['group', 'time']:
        if col in cs.columns:
            cs[col] = pd.to_numeric(cs[col], errors='coerce')
    comparison = dd.merge(cs[[c for c in ['group', 'time', 'att_csdid', 'se_csdid'] if c in cs.columns]], on=['group', 'time'], how='inner')
    comparison['att_difference_diffdiff_minus_csdid'] = comparison['att'] - comparison['att_csdid']
comparison_path = TABLE_DIR / append_tag_to_filename(COMPARISON_NAME, RUN_TAG)
comparison.to_csv(comparison_path, index=False)
print('Saved:', comparison_path)
print(comparison.head(12).to_string(index=False) if not comparison.empty else 'No comparison table created.')

fig_path = FIG_DIR / append_tag_to_filename(EVENT_FIG_NAME, RUN_TAG)
if not diffdiff_gt.empty:
    es = diffdiff_gt.copy()
    es['event_time'] = pd.to_numeric(es['time'], errors='coerce') - pd.to_numeric(es['group'], errors='coerce')
    plot_df = es.groupby(['covariate_set', 'event_time'])['att'].mean().reset_index()
    ax = plot_df.pivot(index='event_time', columns='covariate_set', values='att').plot(figsize=(10, 5), marker='o')
    ax.axhline(0, color='black', linewidth=0.8)
    ax.axvline(-1, color='black', linewidth=0.8, linestyle='--')
    ax.set_title(f'diff-diff C&S event study, {MAIN_BUFFER_KM:g}km stable buffer')
    ax.set_xlabel('Event time')
    ax.set_ylabel('ATT on annual forest loss (m2)')
    ax.grid(axis='y', alpha=0.3)
    ax.xaxis.grid(False)
    ax.legend(frameon=False)
    fig = ax.get_figure()
    fig.tight_layout()
    fig.savefig(fig_path, dpi=220, bbox_inches='tight')
    plt.show()
    print('Saved:', fig_path)

print('Note: package standard errors, csdid comparison output, and the no-covariate spatial-HAC inference layer are saved separately. Covariate-adjusted HAC remains tied to the availability of covariate-adjusted influence functions.')
